# NOOTEBOOK DE CREACION DE DATASET PARA ETIQUETADO

## EN ESTE NOOTEBOOK REALIZAREMOS EL CORTE DE TODOS LOS RASTER POR EL POLIGONO AGRICOLA DE LA LOCALIDAD DE MARCOS JUAREZ.
ASD}

In [ ]:

"""
tile_mosaics.py
Genera chips (PNG) desde mosaics tiffs y produce mapping.csv con:
chip_filename, mosaic_path, row_off, col_off, width, height, transform (affine), crs

Ajusta ROOT_MOSAICS, OUT_CHIPS, tile_size y overlap según necesites.
"""

In [5]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_bounds
import logging
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

def setup_logging(output_dir):
    """Configura el sistema de logging"""
    log_file = os.path.join(output_dir, f'tile_generation_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file, encoding='utf-8'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def calculate_tile_positions(width, height, tile_size, overlap_percent):
    """
    Calcula las posiciones de tiles con solapamiento
    """
    overlap_pixels = int(tile_size * overlap_percent / 100)
    step_size = tile_size - overlap_pixels
    
    # Calcular posiciones en X e Y
    x_positions = []
    y_positions = []
    
    # Posiciones X
    x = 0
    while x < width:
        x_end = min(x + tile_size, width)
        x_positions.append((x, x_end))
        if x_end >= width:
            break
        x += step_size
    
    # Posiciones Y  
    y = 0
    while y < height:
        y_end = min(y + tile_size, height)
        y_positions.append((y, y_end))
        if y_end >= height:
            break
        y += step_size
    
    return x_positions, y_positions

def is_tile_valid(tile_data, threshold_percent=80):
    """
    Verifica si un tile tiene suficiente información útil
    Descarta tiles que sean principalmente negros/vacíos o NaN
    """
    if tile_data is None or tile_data.size == 0:
        return False
    
    # Verificar si hay valores infinitos
    if np.isinf(tile_data).any():
        return False
    
    # Para imágenes de 4 bandas, verificar píxeles válidos (no NaN en TODAS las bandas)
    if len(tile_data.shape) == 3:
        # Píxeles que NO son NaN en todas las bandas
        valid_pixels = ~np.isnan(tile_data).all(axis=0)
        total_pixels = tile_data.shape[1] * tile_data.shape[2]
        valid_count = valid_pixels.sum()
        
        # También verificar que los píxeles válidos no sean todos muy oscuros
        if valid_count > 0:
            valid_data = tile_data[:, valid_pixels]
            dark_pixels = np.sum(np.all(valid_data < 50, axis=0))  # Píxeles muy oscuros en todas las bandas
            dark_percentage_of_valid = (dark_pixels / valid_count) * 100 if valid_count > 0 else 100
        else:
            dark_percentage_of_valid = 100
            
    else:
        # Fallback para imágenes 2D
        valid_pixels = ~np.isnan(tile_data)
        total_pixels = tile_data.size
        valid_count = valid_pixels.sum()
        
        if valid_count > 0:
            dark_pixels = np.sum(tile_data[valid_pixels] < 50)
            dark_percentage_of_valid = (dark_pixels / valid_count) * 100
        else:
            dark_percentage_of_valid = 100
    
    # Calcular porcentaje de píxeles válidos (no NaN)
    valid_percentage = (valid_count / total_pixels) * 100
    
    # Un tile es válido si:
    # 1. Tiene al menos 20% de píxeles válidos (no NaN)
    # 2. Y de esos píxeles válidos, menos del 80% son muy oscuros
    return valid_percentage >= 20 and dark_percentage_of_valid < threshold_percent

def calculate_tile_transform(original_transform, x_start, y_start):
    """
    Calcula la transformación geoespacial para un tile específico
    """
    # Obtener las coordenadas del pixel (x_start, y_start)
    x_coord, y_coord = rasterio.transform.xy(original_transform, y_start, x_start, offset='ul')
    
    # Crear nueva transformación para el tile
    tile_transform = rasterio.transform.from_origin(
        x_coord, y_coord, 
        original_transform.a,  # resolución X
        -original_transform.e  # resolución Y (positiva)
    )
    
    return tile_transform

def generate_tiles_from_mosaic(mosaic_path, output_dir, tile_size=1024, overlap_percent=20, logger=None):
    """
    Genera tiles de un mosaico con solapamiento y conservando metadatos geoespaciales
    """
    if not logger:
        logger = logging.getLogger(__name__)
        
    # Extraer información del nombre del archivo
    filename = os.path.splitext(os.path.basename(mosaic_path))[0]
    # Remover "_mosaic_clipped" del final si existe
    base_name = filename.replace('_mosaic_clipped', '')
    
    logger.info(f"Procesando mosaico: {filename}")
    
    try:
        with rasterio.open(mosaic_path) as src:
            width = src.width
            height = src.height
            bands = src.count
            
            # Verificar si el archivo tiene datos válidos
            logger.info(f"Dimensiones: {width}x{height}, Bandas: {bands}")
            logger.info(f"Tipo de datos: {src.dtypes}")
            logger.info(f"Sistema de coordenadas: {src.crs}")
            logger.info(f"Transform: {src.transform}")
            
            # Leer una muestra más grande para verificar datos
            sample_size = min(500, width, height)  # Muestra más grande para mejor diagnóstico
            sample_data = src.read(window=Window(0, 0, sample_size, sample_size))
            valid_pixels = ~np.isnan(sample_data).all(axis=0)  # Píxeles que no son NaN en todas las bandas
            valid_count = valid_pixels.sum()
            
            if valid_count > 0:
                valid_sample = sample_data[:, valid_pixels]
                logger.info(f"Muestra de datos - Min: {np.nanmin(valid_sample)}, Max: {np.nanmax(valid_sample)}, Píxeles válidos en muestra: {valid_count}/{sample_size*sample_size}")
            else:
                logger.warning(f"La muestra de {sample_size}x{sample_size} píxeles contiene solo NaN - esto puede ser normal si es una zona cortada")
            
            # Verificar que tenga 4 bandas
            if bands != 4:
                logger.warning(f"El mosaico {filename} tiene {bands} bandas, se esperaban 4")
            
            # Calcular posiciones de tiles
            x_positions, y_positions = calculate_tile_positions(width, height, tile_size, overlap_percent)
            
            logger.info(f"Se generarán {len(x_positions)} x {len(y_positions)} = {len(x_positions) * len(y_positions)} tiles")
            
            tiles_created = 0
            tiles_discarded = 0
            
            # Crear tiles
            for row_idx, (y_start, y_end) in enumerate(y_positions):
                for col_idx, (x_start, x_end) in enumerate(x_positions):
                    
                    # Verificar que las coordenadas estén dentro de los límites
                    if x_start >= width or y_start >= height:
                        logger.warning(f"Coordenadas fuera de límites: x={x_start}, y={y_start}, límites: {width}x{height}")
                        tiles_discarded += 1
                        continue
                    
                    # Ajustar coordenadas si exceden los límites
                    x_end_adj = min(x_end, width)
                    y_end_adj = min(y_end, height)
                    
                    # Definir ventana para leer
                    window = Window(x_start, y_start, x_end_adj - x_start, y_end_adj - y_start)
                    
                    # Verificar que la ventana tenga dimensiones válidas
                    if window.width <= 0 or window.height <= 0:
                        logger.warning(f"Ventana inválida: {window}")
                        tiles_discarded += 1
                        continue
                    
                    # Leer datos del tile
                    try:
                        tile_data = src.read(window=window)
                    except Exception as e:
                        logger.error(f"Error leyendo tile r{row_idx:02d}_c{col_idx:02d}: {str(e)}")
                        tiles_discarded += 1
                        continue
                    
                    # Verificar si el tile es válido
                    if not is_tile_valid(tile_data):
                        tiles_discarded += 1
                        continue
                    
                    # Si el tile es más pequeño que el tamaño esperado, rellenarlo con ceros o NoData
                    if tile_data.shape[1] != tile_size or tile_data.shape[2] != tile_size:
                        logger.info(f"Redimensionando tile r{row_idx:02d}_c{col_idx:02d} de {tile_data.shape[1:]} a {tile_size}x{tile_size}")
                        
                        # Crear tile con el tamaño correcto
                        padded_tile = np.full((bands, tile_size, tile_size), np.nan, dtype=tile_data.dtype)
                        padded_tile[:, :tile_data.shape[1], :tile_data.shape[2]] = tile_data
                        tile_data = padded_tile
                    
                    # Generar nombre del tile
                    tile_name = f"{base_name}_tile_r{row_idx:02d}_c{col_idx:02d}.tif"
                    tile_path = os.path.join(output_dir, tile_name)
                    
                    # Calcular transformación geoespacial del tile
                    tile_transform = calculate_tile_transform(src.transform, x_start, y_start)
                    
                    # Configurar metadatos del tile
                    tile_meta = src.meta.copy()
                    tile_meta.update({
                        'height': tile_data.shape[1],
                        'width': tile_data.shape[2],
                        'transform': tile_transform
                    })
                    
                    # Guardar tile
                    try:
                        with rasterio.open(tile_path, 'w', **tile_meta) as dst:
                            dst.write(tile_data)
                            # Copiar también información de coordenadas si existe
                            if src.crs:
                                dst.crs = src.crs
                        
                        # Verificar que el archivo se guardó correctamente leyendo una pequeña muestra
                        with rasterio.open(tile_path) as verify:
                            verify_sample = verify.read(window=Window(0, 0, min(10, tile_size), min(10, tile_size)))
                            if np.isnan(verify_sample).all():
                                logger.warning(f"El tile guardado {tile_name} contiene solo NaN en la muestra verificada")
                            # No eliminar el archivo ya que puede ser normal tener tiles con mucho NaN
                        
                        tiles_created += 1
                        
                    except Exception as e:
                        logger.error(f"Error guardando tile {tile_name}: {str(e)}")
                        tiles_discarded += 1
                        continue
            
            logger.info(f"Mosaico {filename}: {tiles_created} tiles creados, {tiles_discarded} descartados")
            return tiles_created, tiles_discarded
            
    except Exception as e:
        logger.error(f"Error procesando {mosaic_path}: {str(e)}")
        return 0, 0

def main():
    """Función principal"""
    # Configuración
    input_dir = r"E:\Silos\Base de datos\procesado_sin_nubes\mosaics\mosaics_clipped"
    output_dir = r"E:\Silos\Base de datos\procesado_sin_nubes\tiles"
    
    tile_size = 1024  # píxeles
    overlap_percent = 20  # porcentaje de solapamiento
    
    # Crear directorio de salida si no existe
    os.makedirs(output_dir, exist_ok=True)
    
    # Configurar logging
    logger = setup_logging(output_dir)
    
    logger.info("="*50)
    logger.info("INICIO DEL PROCESO DE GENERACIÓN DE TILES")
    logger.info("="*50)
    logger.info(f"Directorio de entrada: {input_dir}")
    logger.info(f"Directorio de salida: {output_dir}")
    logger.info(f"Tamaño de tile: {tile_size}x{tile_size} píxeles")
    logger.info(f"Solapamiento: {overlap_percent}%")
    
    # Buscar archivos TIFF en el directorio
    tiff_files = [f for f in os.listdir(input_dir) if f.lower().endswith('.tif')]
    
    if not tiff_files:
        logger.error("No se encontraron archivos TIFF en el directorio de entrada")
        return
    
    logger.info(f"Se encontraron {len(tiff_files)} archivos TIFF para procesar")
    
    total_tiles_created = 0
    total_tiles_discarded = 0
    processed_files = 0
    
    # Procesar cada archivo
    for filename in tqdm(tiff_files, desc="Procesando mosaicos"):
        mosaic_path = os.path.join(input_dir, filename)
        
        try:
            tiles_created, tiles_discarded = generate_tiles_from_mosaic(
                mosaic_path, output_dir, tile_size, overlap_percent, logger
            )
            
            total_tiles_created += tiles_created
            total_tiles_discarded += tiles_discarded
            processed_files += 1
            
        except Exception as e:
            logger.error(f"Error procesando archivo {filename}: {str(e)}")
            continue
    
    # Resumen final
    logger.info("="*50)
    logger.info("RESUMEN FINAL")
    logger.info("="*50)
    logger.info(f"Archivos procesados: {processed_files}/{len(tiff_files)}")
    logger.info(f"Total tiles creados: {total_tiles_created}")
    logger.info(f"Total tiles descartados: {total_tiles_discarded}")
    logger.info(f"Tiles guardados en: {output_dir}")
    
    if total_tiles_created > 0:
        avg_tiles_per_mosaic = total_tiles_created / processed_files
        logger.info(f"Promedio de tiles válidos por mosaico: {avg_tiles_per_mosaic:.1f}")
    
    logger.info("Proceso completado exitosamente!")

if __name__ == "__main__":
    main()

2025-09-03 20:06:59,192 - INFO - ==================================================
2025-09-03 20:06:59,195 - INFO - INICIO DEL PROCESO DE GENERACIÓN DE TILES
2025-09-03 20:06:59,197 - INFO - ==================================================
2025-09-03 20:06:59,199 - INFO - Directorio de entrada: E:\Silos\Base de datos\procesado_sin_nubes\mosaics\mosaics_clipped
2025-09-03 20:06:59,200 - INFO - Directorio de salida: E:\Silos\Base de datos\procesado_sin_nubes\tiles
2025-09-03 20:06:59,201 - INFO - Tamaño de tile: 1024x1024 píxeles
2025-09-03 20:06:59,202 - INFO - Solapamiento: 20%
2025-09-03 20:06:59,205 - INFO - Se encontraron 328 archivos TIFF para procesar


Procesando mosaicos:   0%|          | 0/328 [00:00<?, ?it/s]2025-09-03 20:06:59,210 - INFO - Procesando mosaico: 2020_6_10_0_mosaic_clipped
2025-09-03 20:06:59,234 - INFO - Dimensiones: 9175x20982, Bandas: 4
2025-09-03 20:06:59,236 - INFO - Tipo de datos: ('float32', 'float32', 'float32', 'float32')
2025-09-03 20:06:59,237 - INFO - Sistema de coordenadas: EPSG:32720
2025-09-03 20:06:59,239 - INFO - Transform: | 10.00, 0.00, 518030.00|
| 0.00,-10.00, 6453870.00|
| 0.00, 0.00, 1.00|
2025-09-03 20:06:59,410 - WARNING - La muestra de 500x500 píxeles contiene solo NaN - esto puede ser normal si es una zona cortada
2025-09-03 20:06:59,411 - INFO - Se generarán 11 x 26 = 286 tiles
2025-09-03 20:07:03,169 - WARNING - El tile guardado 2020_6_10_0_tile_r06_c05.tif contiene solo NaN en la muestra verificada
2025-09-03 20:07:03,642 - WARNING - El tile guardado 2020_6_10_0_tile_r06_c07.tif contiene solo NaN en la muestra verificada
2025-09-03 20:07:06,925 - WARNING - El tile guardado 2020_6_10_0_ti

In [2]:
import rasterio
import numpy as np

# Prueba con uno de tus mosaicos
mosaic_path = r"E:\Silos\Base de datos\procesado_sin_nubes\mosaics\mosaics_clipped\2020_6_10_0_mosaic_clipped.tif"

with rasterio.open(mosaic_path) as src:
    # Leer todo el archivo
    data = src.read()
    print(f"Shape: {data.shape}")
    print(f"Data type: {data.dtype}")
    print(f"Min: {np.nanmin(data)}")
    print(f"Max: {np.nanmax(data)}")
    print(f"NaN count: {np.isnan(data).sum()}")
    print(f"Total pixels: {data.size}")
    print(f"Percentage NaN: {np.isnan(data).sum() / data.size * 100:.1f}%")

Shape: (4, 20982, 9175)
Data type: float32
Min: 1.0
Max: 17728.0
NaN count: 528558020
Total pixels: 770039400
Percentage NaN: 68.6%
